# 04 · RAG evaluation — ranking, refusal and error analysis

> **Core lesson:** all cells execute without external services. Read the *predict → run → explain* questions before each experiment.


## Measure the stages, not an invented 'RAG accuracy'

Document retrieval metrics are different from final answer correctness. Our `gold_doc_ids` dataset evaluates whether the target source appears among top-k hits.

- **hit@k**: share of answerable questions whose labeled document appears in top-k.
- **MRR@k**: average `1 / rank` of first relevant retrieved source (0 when absent).
- **unanswerable abstention**: fraction of labeled unknown questions with no score above threshold.
- **Faithfulness**: percentage of generated claims supported by cited spans — **NOT computed** by this offline baseline.

The gold set has 10 answerable and 2 unanswerable examples. This small synthetic set is an illustration, **not** a production benchmark.


In [ ]:
from pathlib import Path
from RAG.src.core import (load_documents, chunk_documents, load_questions,
                          Retriever, evaluate_retrieval)
data = Path("RAG/data")
docs = load_documents(data / "support_policies.jsonl")
questions = load_questions(data / "evaluation_questions.jsonl")
retriever = Retriever(chunk_documents(docs))
print("Evaluation questions:", len(questions))
print("Positive examples:", sum(bool(r["gold_doc_ids"]) for r in questions))
print("Unknown examples:", sum(not r["gold_doc_ids"] for r in questions))


## Baseline evaluation with explicit denominators

An apparent metric improvement can be deceptive if you change both the documents and questions. Keep a fixed holdout. This demo's source IDs represent complete, short policies.


In [ ]:
for k in (1, 3, 5):
    values = evaluate_retrieval(retriever, questions, k=k, threshold=0.12)
    print(f"k={k} -> {values}")
    assert 0 <= values["hit_at_k"] <= 1


## Chunk size sensitivity

Short chunks can lose a critical nearby condition while still matching a query. Large chunks can fill an LLM context with irrelevant policy details. Which chunk setting gives the most useful *evidence* instead of the best-looking retrieval score?


In [ ]:
for size, overlap in [(12, 3), (24, 6), (80, 15)]:
    index = Retriever(chunk_documents(docs, size=size, overlap=overlap))
    values = evaluate_retrieval(index, questions, k=3)
    print(f"size={size:>2}, overlap={overlap:>2}: hit@3={values['hit_at_k']}, MRR@3={values['mrr_at_k']}")


## Inspect errors, not only averages

Show one retrieved source versus expected evidence. Students should explain **whether a failure came from missing knowledge, bad parsing/chunking, indexing/ranking, generator reasoning or an unsupported citation**.


In [ ]:
for row in questions[:5]:
    hits = retriever.search(row["question"], 2)
    predicted = [h.chunk.doc_id for h in hits if h.score >= 0.12]
    print("Question:", row["question"])
    print("Expected:", row["gold_doc_ids"], "| Ranked:", predicted)
    print("---")


## Release gate and limitations

Check **retrieval recall**, **citation faithfulness**, **refusal correctness**, and a human assessment of answer quality. The evaluated offline preview is deterministic but is *not* a generator hallucination test. A model swap, changed prompt or upgraded corpus mandates fresh end-to-end evaluation.

**Change → effect:** increase threshold to 0.4. Is refusal improved at the cost of recall? Plot both against a threshold sweep and select a business-appropriate operating point.

Read [evaluation and production operations](../docs/evaluation_and_operations.md); proceed to **05_capstone**.
